In [1]:
import pandas as pd
import numpy as np
import re
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

In [3]:
# Load the datasets
train_df = pd.read_csv('train_(2)_(1)_(1).csv')
dist_df = pd.read_csv('dist_from_city_centre_(1)_(1)_(1).csv')
rent_df = pd.read_csv('avg_rent_(1)_(1)_(1).csv')
test_df = pd.read_csv('test_(2)_(1)_(1).csv')

In [5]:
# Clean location names in all datasets (remove spaces, lowercase)
train_df['location'] = train_df['location'].str.strip().str.lower()
test_df['location'] = test_df['location'].str.strip().str.lower()
dist_df['location'] = dist_df['location'].str.strip().str.lower()
rent_df['location'] = rent_df['location'].str.strip().str.lower()

In [7]:
# Merge dist_from_city and avg_2bhk_rent into training data
train_df = train_df.merge(dist_df, on='location', how='left')
train_df = train_df.merge(rent_df, on='location', how='left')

In [9]:
# Merge into test data
test_df = test_df.merge(dist_df, on='location', how='left')
test_df = test_df.merge(rent_df, on='location', how='left')

In [11]:
# Handle missing values for new columns
train_df['dist_from_city'] = train_df['dist_from_city'].fillna(train_df['dist_from_city'].median())
train_df['avg_2bhk_rent'] = train_df['avg_2bhk_rent'].fillna(train_df['avg_2bhk_rent'].median())
test_df['dist_from_city'] = test_df['dist_from_city'].fillna(train_df['dist_from_city'].median())  # Use train median
test_df['avg_2bhk_rent'] = test_df['avg_2bhk_rent'].fillna(train_df['avg_2bhk_rent'].median())  # Use train median

In [13]:
# Function to clean total_sqft
def clean_total_sqft(sqft):
    try:
        sqft = str(sqft).strip()
        if sqft.replace('.', '').isdigit():
            return float(sqft)
        if '-' in sqft:
            start, end = map(float, sqft.split('-'))
            return (start + end) / 2
        match = re.match(r'(\d+\.?\d*)\s*([A-Za-z\s\.]+)?', sqft)
        if match:
            value = float(match.group(1))
            unit = match.group(2).lower().replace(' ', '') if match.group(2) else ''
            if unit in ['sq.meter', 'sqmeter', 'sq.m', 'sqm']:
                return value * 10.7639
            elif unit in ['sq.yards', 'sqyard', 'sq.y', 'sqy']:
                return value * 9
            elif unit in ['acres', 'acre']:
                return value * 43560
            elif unit in ['perch']:
                return value * 272.25
            elif unit in ['cents', 'cent']:
                return value * 435.6
            elif unit in ['guntha']:
                return value * 1089
            elif unit == '' or unit in ['sq.ft', 'sqft', 'squarefeet']:
                return value
        return np.nan
    except:
        return np.nan

In [15]:
# Apply total_sqft cleaning
train_df['total_sqft'] = train_df['total_sqft'].apply(clean_total_sqft)
train_df['total_sqft'] = train_df['total_sqft'].fillna(train_df['total_sqft'].median())
test_df['total_sqft'] = test_df['total_sqft'].apply(clean_total_sqft)
test_df['total_sqft'] = test_df['total_sqft'].fillna(train_df['total_sqft'].median())  # Use train median

In [17]:
# Standardize size to "X BHK"
def standardize_size(size):
    try:
        if pd.isna(size) or not isinstance(size, str) or size.strip() == '':
            return np.nan
        size = size.strip().lower()
        match = re.match(r'(\d+)', size)
        if match:
            number = match.group(1)
            return f"{number} BHK"
        if 'studio' in size:
            return "1 BHK"
        if 'rk' in size:
            return f"{size[0]} BHK"
        return np.nan
    except:
        return np.nan

In [19]:
train_df['size'] = train_df['size'].apply(standardize_size)
train_df['size'] = train_df['size'].fillna(train_df['size'].mode()[0])
test_df['size'] = test_df['size'].apply(standardize_size)
test_df['size'] = test_df['size'].fillna(train_df['size'].mode()[0])  # Use train mode

In [21]:
# Handle other missing values
train_df['bath'] = train_df['bath'].fillna(train_df['bath'].median())
train_df['balcony'] = train_df['balcony'].fillna(train_df['balcony'].median())
train_df['society'] = train_df['society'].fillna('Unknown')
train_df['location'] = train_df['location'].fillna(train_df['location'].mode()[0])
test_df['bath'] = test_df['bath'].fillna(train_df['bath'].median())  # Use train median
test_df['balcony'] = test_df['balcony'].fillna(train_df['balcony'].median())  # Use train median
test_df['society'] = test_df['society'].fillna('Unknown')
test_df['location'] = test_df['location'].fillna(train_df['location'].mode()[0])  # Use train mode

In [23]:
# Simplify availability
train_df['availability'] = train_df['availability'].apply(lambda x: 'Ready' if x == 'Ready To Move' else 'Not Ready')
test_df['availability'] = test_df['availability'].apply(lambda x: 'Ready' if x == 'Ready To Move' else 'Not Ready')

In [25]:
# Drop unnecessary columns
train_df = train_df.drop(['ID', 'society'], axis=1)
test_ids = test_df['ID']  # Save IDs for predictions
test_df = test_df.drop(['ID', 'society'], axis=1)

In [27]:
# Encode categorical variables
categorical_cols = ['area_type', 'availability', 'location', 'size']
train_df = pd.get_dummies(train_df, columns=categorical_cols, drop_first=True)
test_df = pd.get_dummies(test_df, columns=categorical_cols, drop_first=True)

In [29]:
# Align test data columns with training data
test_df = test_df.reindex(columns=train_df.columns.drop('price'), fill_value=0)

In [31]:
# Define features and target
X_train = train_df.drop('price', axis=1)
y_train = train_df['price']
X_test = test_df  # Test data has no price column

In [33]:
# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [35]:
# Train Linear Regression
model = LinearRegression()
model.fit(X_train_scaled, y_train)

LinearRegression()

In [37]:
# Predict on test data
test_predictions = model.predict(X_test_scaled)

In [43]:
# Save predictions to CSV
predictions_df = pd.DataFrame({'ID': test_ids, 'price': test_predictions})
predictions_df.to_csv('test_predictions.csv', index=False)
print("Predictions saved to 'test_predictions.csv'")

Predictions saved to 'test_predictions.csv'
